# 07 -- OpenAssetPricing Data Collection

## Purpose
Filters the OpenAssetPricing (OAP) monthly factor dataset down to only the PERMNOs in the top-100 S&P 500 universe. OAP already uses PERMNO as its identifier, so no linking table is needed -- the master list is inner-joined directly on `permno`.

## Source
OpenAssetPricing signed predictors dataset, originally downloaded as `signed_predictors_dl_wide.csv` (~8.5 GB, ~5.4M rows x 211 columns). The CSV is first converted to parquet using DuckDB for efficient downstream querying.

## Input
- `Data/Data_Collection/Initial/07_OpenAssetPricing/signed_predictors_dl_wide.csv` -- raw OAP download
- `Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` -- master PERMNO list from notebook 05

## Collection Method
DuckDB is used to filter the large OAP file without loading the full dataset into memory. The master PERMNO list is registered as an in-memory DuckDB table and inner-joined against the parquet file on disk, filtered to `yyyymm >= 200401 AND yyyymm <= 202412`. This takes approximately 30-60 seconds depending on disk speed.

## Variables Collected
All ~207 factor columns from OAP are retained (the full set of signed predictors). No factor selection is performed at this stage. The `yyyymm` integer column (e.g., 202401 for January 2024) is converted to a proper month-end datetime as `date`, and a `year` column is added for parquet partitioning.

## Output
`Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly/` -- partitioned parquet by year (subfolders `year=2004/`, `year=2005/`, etc.)

In [ ]:
import duckdb as db

db.query("""
    COPY (SELECT * FROM '../../Data/Data_Collection/Initial/07_OpenAssetPricing/signed_predictors_dl_wide.csv') 
    TO 'factors.parquet' (FORMAT PARQUET)
""")


# %% [markdown]
# # Stage 3: Collect and Filter OpenAssetPricing Data
#
# This notebook filters the OpenAssetPricing (OAP) monthly factor dataset down
# to only the PERMNOs in our top-100 S&P 500 universe.
#
# OAP already uses PERMNO as its identifier, so no linking table is needed —
# we simply inner-join on permno with our master list.
#
# The raw OAP file is ~8.5 GB (5.4M rows × 211 columns). We use DuckDB to
# filter it efficiently without loading the entire file into memory.
#
# Output: `data/firm_monthly/` — partitioned parquet by year.

# %% [markdown]
# ## Setup

# %%
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path
import shutil
import gc



# %% [markdown]
# ## Step 1: Load the master PERMNO list

# %%
master = pd.read_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
print(f"Master list: {len(master)} unique PERMNOs")
print(f"Sample PERMNOs: {master['permno'].head(10).tolist()}")

# %% [markdown]
# ## Step 2: Filter OAP to our universe using DuckDB
#
# DuckDB can read parquet files directly and filter them without loading
# the full 8.5 GB into pandas. We register the master list as an in-memory
# table and inner-join against the parquet file on disk.
#
# This should take ~30-60 seconds depending on disk speed.

# %%
con = duckdb.connect()

# Register the master PERMNO list as a DuckDB table
con.register('master', master)

# Check the raw OAP file dimensions first
raw_info = con.execute("""
    SELECT COUNT(*) as n_rows, 
           COUNT(DISTINCT permno) as n_permnos,
           MIN(yyyymm) as min_date,
           MAX(yyyymm) as max_date
    FROM 'factors.parquet'
""").fetchdf()
print("Raw OAP file:")
print(f"  Rows: {raw_info['n_rows'].iloc[0]:,}")
print(f"  Unique PERMNOs: {raw_info['n_permnos'].iloc[0]:,}")
print(f"  Date range: {raw_info['min_date'].iloc[0]} to {raw_info['max_date'].iloc[0]}")

# %%
# Filter to our universe and date range
print("\nFiltering OAP to master list PERMNOs and date range 200401-202412...")

df = con.execute("""
    SELECT f.*
    FROM 'factors.parquet' f
    INNER JOIN master m ON f.permno = m.permno
    WHERE f.yyyymm >= 200401 AND f.yyyymm <= 202412
""").fetchdf()

con.close()

print(f"Filtered OAP: {len(df):,} rows, {df['permno'].nunique()} unique PERMNOs")
print(f"Columns: {len(df.columns)}")

# %% [markdown]
# ## Step 3: Create proper date columns
#
# The OAP data uses `yyyymm` as an integer (e.g., 202401 for January 2024).
# We convert this to a proper month-end datetime for consistency with other datasets,
# and add a `year` column for parquet partitioning.

# %%
df['date'] = pd.to_datetime(df['yyyymm'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
df['year'] = df['date'].dt.year

# Sort for clean output
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Shape: {df.shape}")

# %% [markdown]
# ## Step 4: Save as partitioned parquet

# %%
if Path('../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly').exists():
    shutil.rmtree('../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly')

df.to_parquet('../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly', partition_cols=['year'], engine='pyarrow', index=False)

year_folders = sorted(Path('../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly').glob('year=*'))
print(f"Saved to ../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly with {len(year_folders)} year partitions")
print(f"First: {year_folders[0].name}, Last: {year_folders[-1].name}")

# %% [markdown]
# ## Step 5: Verification & Summary Stats

# %% [markdown]
# ### Basic dimensions

# %%
print(f"Total rows: {len(df):,}")
print(f"Unique PERMNOs: {df['permno'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total columns (incl permno, yyyymm, date, year): {len(df.columns)}")
print(f"Factor columns: {len(df.columns) - 4}")  # minus permno, yyyymm, date, year

# %% [markdown]
# ### Rows per year
#
# Expect ~150-200 PERMNOs × 12 months = ~1,800-2,400 rows per year.

# %%
rows_per_year = df.groupby('year').size()
print("Rows per year:")
print(rows_per_year.to_string())
print(f"\nMean: {rows_per_year.mean():,.0f}, Min: {rows_per_year.min():,}, Max: {rows_per_year.max():,}")

# %% [markdown]
# ### Factor coverage analysis
#
# Many OAP factors are sparse — they only exist for certain firms or time periods.
# This is expected. We identify the best-covered and worst-covered factors.

# %%
# Exclude non-factor columns
meta_cols = ['permno', 'yyyymm', 'date', 'year']
factor_cols = [c for c in df.columns if c not in meta_cols]

non_null_pct = ((df[factor_cols].notna().sum() / len(df)) * 100).round(1)

print(f"\n--- 15 BEST covered factors (highest % non-null) ---")
best = non_null_pct.sort_values(ascending=False).head(15)
for name, pct in best.items():
    print(f"  {name:40s} {pct:5.1f}%")

print(f"\n--- 15 WORST covered factors (lowest % non-null) ---")
worst = non_null_pct.sort_values(ascending=True).head(15)
for name, pct in worst.items():
    print(f"  {name:40s} {pct:5.1f}%")

print(f"\n--- Coverage distribution ---")
print(f"  Factors with >90% coverage: {(non_null_pct > 90).sum()}")
print(f"  Factors with 50-90% coverage: {((non_null_pct > 50) & (non_null_pct <= 90)).sum()}")
print(f"  Factors with 10-50% coverage: {((non_null_pct > 10) & (non_null_pct <= 50)).sum()}")
print(f"  Factors with <10% coverage: {(non_null_pct < 10).sum()}")

# %% [markdown]
# ### Check which master PERMNOs are missing from OAP
#
# Some PERMNOs in our master list may not appear in OAP at all
# (e.g., if the stock was added to S&P 500 very recently and OAP
# hasn't updated, or if it's a share class that OAP doesn't cover).

# %%
oap_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'].tolist())
missing = master_permnos - oap_permnos

if missing:
    print(f"WARNING: {len(missing)} PERMNOs from master list not in OAP:")
    print(f"  {sorted(missing)}")
else:
    print("All master list PERMNOs found in OAP data.")

present = master_permnos & oap_permnos
print(f"\nCoverage: {len(present)}/{len(master_permnos)} "
      f"({len(present)/len(master_permnos)*100:.1f}%) of master PERMNOs present in OAP")

# %% [markdown]
# ### Sample data for one stock
#
# Show a few rows and a handful of well-known factors for a recognisable PERMNO.

# %%
# Use AAPL (14593) or fall back to the first PERMNO if not present
sample_permno = 14593 if 14593 in oap_permnos else df['permno'].iloc[0]

sample = df[df['permno'] == sample_permno].tail(6)
sample_cols = ['permno', 'date', 'Size', 'BM', 'Mom12m', 'GP', 'roaq', 
               'Beta', 'IdioVol3F', 'Illiquidity', 'EP']
# Only show columns that actually exist
sample_cols = [c for c in sample_cols if c in df.columns]

print(f"Sample data for PERMNO {sample_permno} (last 6 months):")
print(sample[sample_cols].to_string(index=False))

# %% [markdown]
# ## Cleanup

# %%
del df
gc.collect()

print("\nStage 3 complete. File saved:")
print("  ../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly/ (partitioned parquet by year)")